In [1]:
import pandas as pd
import sqlite3
import os

df_t = pd.read_csv('../data/transactions_clean.csv')
df_e = pd.read_csv('../data/employes_clean.csv')

conn = sqlite3.connect('../outputs/finance.db')
print("Base de données finance.db créée")


# Dimension types de transactions
dim_type = df_t[['type']].drop_duplicates().reset_index(drop=True)
dim_type['type_id'] = dim_type.index + 1
dim_type = dim_type[['type_id', 'type']]
dim_type.to_sql('dim_type_transaction', conn, if_exists='replace', index=False)
print(f"dim_type_transaction : {len(dim_type)} lignes")

# Dimension départements
dim_dept = df_e[['departement', 'unite']].drop_duplicates().reset_index(drop=True)
dim_dept['dept_id'] = dim_dept.index + 1
dim_dept = dim_dept[['dept_id', 'departement', 'unite']]
dim_dept.to_sql('dim_departement', conn, if_exists='replace', index=False)
print(f"dim_departement : {len(dim_dept)} lignes")

# Dimension postes
dim_poste = df_e[['poste']].drop_duplicates().reset_index(drop=True)
dim_poste['poste_id'] = dim_poste.index + 1
dim_poste = dim_poste[['poste_id', 'poste']]
dim_poste.to_sql('dim_poste', conn, if_exists='replace', index=False)
print(f"dim_poste : {len(dim_poste)} lignes")

# Dimension villes
dim_ville = df_e[['ville']].drop_duplicates().reset_index(drop=True)
dim_ville['ville_id'] = dim_ville.index + 1
dim_ville = dim_ville[['ville_id', 'ville']]
dim_ville.to_sql('dim_ville', conn, if_exists='replace', index=False)
print(f"dim_ville : {len(dim_ville)} lignes")


# Jointure pour récupérer les IDs
df_t_fact = df_t.merge(dim_type, on='type', how='left')

# Table faits transactions
fact_transactions = df_t_fact[[
    'type_id', 'step', 'amount',
    'compte_source', 'solde_avant_source', 'solde_apres_source',
    'compte_dest', 'solde_avant_dest', 'solde_apres_dest',
    'est_fraude', 'est_flagge'
]].copy()

fact_transactions['transaction_id'] = range(1, len(fact_transactions) + 1)
fact_transactions = fact_transactions[['transaction_id'] + [c for c in fact_transactions.columns if c != 'transaction_id']]

fact_transactions.to_sql('fact_transactions', conn, if_exists='replace', index=False)
print(f"fact_transactions : {len(fact_transactions)} lignes")

# Jointure pour récupérer les IDs dimensions
df_e_fact = df_e.merge(dim_dept, on=['departement', 'unite'], how='left')
df_e_fact = df_e_fact.merge(dim_poste, on='poste', how='left')
df_e_fact = df_e_fact.merge(dim_ville, on='ville', how='left')

# Table faits employés
fact_employes = df_e_fact[[
    'employe_id', 'dept_id', 'poste_id', 'ville_id',
    'annee', 'age', 'anciennete',
    'genre', 'statut', 'raison_depart', 'type_depart',
    'date_embauche', 'date_depart'
]].copy()

fact_employes.to_sql('fact_employes', conn, if_exists='replace', index=False)
print(f"fact_employes : {len(fact_employes)} lignes")


# Lister toutes les tables créées
cursor = conn.cursor()
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables dans finance.db :")
for t in tables:
    nom = t[0]
    count = cursor.execute(f"SELECT COUNT(*) FROM {nom}").fetchone()[0]
    print(f"  {nom} : {count} lignes")

conn.close()
print("\nConnexion fermée — base prête")

Base de données finance.db créée
dim_type_transaction : 5 lignes
dim_departement : 21 lignes
dim_poste : 47 lignes
dim_ville : 40 lignes
fact_transactions : 6362604 lignes
fact_employes : 49653 lignes
Tables dans finance.db :
  dim_type_transaction : 5 lignes
  dim_departement : 21 lignes
  dim_poste : 47 lignes
  dim_ville : 40 lignes
  fact_transactions : 6362604 lignes
  fact_employes : 49653 lignes

Connexion fermée — base prête
